<a href="https://colab.research.google.com/github/NourEldin-Osama/AI-Notebooks/blob/main/CodingAgents/OpenCode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Install ngrok
!curl -sSL https://ngrok-agent.s3.amazonaws.com/ngrok.asc \
  | sudo tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null \
  && echo "deb https://ngrok-agent.s3.amazonaws.com bookworm main" \
  | sudo tee /etc/apt/sources.list.d/ngrok.list \
  && sudo apt update \
  && sudo apt install ngrok

In [ ]:
# @title Setup OpenCode, Ngrok (CLI Version)
import os
import time
from google.colab import userdata

# Install httpx
!uv pip install -q httpx
import httpx

# 1. Authenticate your downloaded ngrok CLI
# Get a free authtoken at https://ngrok.com and save it in Colab Secrets as 'NGROK_AUTHTOKEN'
NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")
!ngrok config add-authtoken $NGROK_AUTHTOKEN

# 2. Install OpenCode
!curl -fsSL https://opencode.ai/install | bash
os.environ['PATH'] = '/root/.opencode/bin:' + os.environ['PATH']

# 3. Start the downloaded ngrok in the background
# We pipe the output to /dev/null so it doesn't clutter the cell output
print("⏳ Starting ngrok tunnel in the background...")
os.system("ngrok http 4096 > /dev/null 2>&1 &")

# Give ngrok a few seconds to connect to the servers
time.sleep(3) 

# Fetch the public URL from ngrok's local API using httpx (it runs on port 4040 by default)
try:
    response = httpx.get("http://localhost:4040/api/tunnels")
    response.raise_for_status() # This will throw an error if the HTTP request failed
    
    public_url = response.json()["tunnels"][0]["public_url"]
    print("\n" + "="*55)
    print("✅ Ngrok tunnel established successfully!")
    print(f"🚀 Your OpenCode Web UI URL: {public_url}")
    print("="*55 + "\n")
    
except Exception as e:
    print("\n❌ Failed to retrieve ngrok URL.")
    print(f"Error details: {e}")
    print("💡 Hint: Double-check your NGROK_AUTHTOKEN in Colab Secrets.\n")

print("💻 To use the OpenCode CLI, open the Colab terminal and type:")
print("   >>> opencode")
print("\n🌐 Spinning up OpenCode web server on port 4096...\n")

# 4. Start Opencode Web
!opencode web --hostname 0.0.0.0 --port 4096 -- /content